# ParkCast SF — LightGBM Training (Accuracy-Optimized)

Trains a LightGBM residual model on real paid-meter rows only.

**Key design choices:**
- Temporal 80/20 split (no shuffle)
- Block-level historical aggregates (from training rows only — no lookahead)
- Residual target: `y − block×hour×dow_mean`
- LightGBM with early stopping on validation MAE
- Event-row upweighting (20×)

## Imports

In [1]:
import os
import json
import socket
import urllib.parse
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
from sklearn.metrics import mean_absolute_error, r2_score

/opt/homebrew/anaconda3/lib/python3.13/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


## Configuration

In [2]:
PROJECT_DIR = os.path.dirname(os.getcwd())  # project root (run notebook from dev/)
DATA_DIR = os.path.join(PROJECT_DIR, "data")
DATA_PATH = os.path.join(DATA_DIR, "processed_training_data.csv")
CIT_PATH = os.path.join(DATA_DIR, "citations_by_block.csv")
MODEL_DIR = os.path.join(PROJECT_DIR, "app", "models")
MODEL_PATH = os.path.join(MODEL_DIR, "LightGBM.pkl")
META_PATH = os.path.join(MODEL_DIR, "LightGBM.meta.json")
BLOCK_AGG_PATH = os.path.join(MODEL_DIR, "LightGBM.block_aggs.parquet")

# MLflow tracking — remote server on GCP. Override via env for local runs.
# If the server is unreachable (down, offline, firewall), fall back to a
# local file store so training still completes and produces a run record
# we can backfill to the server later.
MLFLOW_TRACKING_URI = os.environ.get(
    "MLFLOW_TRACKING_URI", "http://34.134.159.106:5000"
)
# v2: the v1 experiment was created against the old server config with a
# local-filesystem artifact root. Post-restart with --artifacts-destination,
# we need a fresh experiment so runs inherit the GCS artifact location.
MLFLOW_EXPERIMENT = os.environ.get("MLFLOW_EXPERIMENT", "parkcast-sf-lightgbm-v2")
MLFLOW_LOCAL_FALLBACK = f"file:{os.path.join(PROJECT_DIR, 'mlruns')}"
# Name in the MLflow model registry. Each training run adds a new version;
# @champion alias points at the best version (smart-promote at end of run).
REGISTERED_MODEL_NAME = "parkcast-sf-lightgbm"


def _mlflow_reachable(uri, timeout=2):
    try:
        u = urllib.parse.urlparse(uri)
        port = u.port or (443 if u.scheme == "https" else 80)
        with socket.create_connection((u.hostname, port), timeout=timeout):
            return True
    except OSError:
        return False


if _mlflow_reachable(MLFLOW_TRACKING_URI):
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    print(f"MLflow: tracking → {MLFLOW_TRACKING_URI}")
else:
    mlflow.set_tracking_uri(MLFLOW_LOCAL_FALLBACK)
    MLFLOW_TRACKING_URI = MLFLOW_LOCAL_FALLBACK
    print(f"MLflow: server unreachable — logging to {MLFLOW_LOCAL_FALLBACK}")
mlflow.set_experiment(MLFLOW_EXPERIMENT)

TARGET = "occupancy_pct"
BASELINE_COL = "block_hour_dow_mean"
FEATURES_NUMERIC = [
    "hour", "day_of_week", "month", "is_weekend", "is_holiday",
    "is_school_day", "is_raining", "temperature", "event_intensity",
    "citation_count", "citations_hourly_median",
    "complaints_311_median", "complaints_311_total",
    "poi_dining_200m", "poi_retail_200m",
    "poi_transit_200m", "poi_attraction_200m",
    "sfpark_block_occ",
    "permit_active", "permit_count_30d",
    "lat", "lon", "total_spaces",
    "block_mean", "block_hour_mean",
    "lag_1d", "lag_2d", "lag_7d", "lag_14d", "lag_28d",
    "lag_3d_mean", "lag_7d_mean",
]
FEATURES_CATEGORICAL = ["neighborhood"]
FEATURES = FEATURES_NUMERIC + FEATURES_CATEGORICAL

# Point-lag offsets (hours) kept as features.
LAG_HOURS = [1 * 24, 2 * 24, 7 * 24, 14 * 24, 28 * 24]
# Rolling-mean lags: feature_name → number of prior days averaged at the same
# (block, timestamp). Smooths single-day noise that point lags inherit.
LAG_ROLLING_FEATURES = {"lag_3d_mean": 3, "lag_7d_mean": 7}

# Exponential recency decay applied to training sample weights. A row `d` days
# before the end of the training window gets weight 0.5 ** (d / HALF_LIFE),
# so a 90-day half-life gives the last month ~2.2× the weight of 3 months ago,
# ~5× the weight of 6 months ago. Test set is always the most-recent 20%, so
# aligning train weights with recency matches what we evaluate on.
# Set to None to disable.
RECENCY_HALF_LIFE_DAYS = 90.0

MLflow: tracking → http://34.134.159.106:5000


## `temporal_split()`

In [3]:
def temporal_split(df, frac=0.80):
    df = df.sort_values("timestamp").reset_index(drop=True)
    split_time = df["timestamp"].quantile(frac)
    return df[df["timestamp"] < split_time].copy(), df[df["timestamp"] >= split_time].copy(), split_time

## `refit_citations_median()`

In [4]:
def refit_citations_median(train, test, split_time):
    if not os.path.exists(CIT_PATH):
        return train, test
    cit = pd.read_csv(CIT_PATH, parse_dates=["timestamp"])
    cit = cit[cit["timestamp"] < split_time].copy()
    cit["hour_of_day"] = cit["timestamp"].dt.hour
    cit["weekday"] = cit["timestamp"].dt.weekday
    med = (cit.groupby(["hour_of_day", "weekday"])["citation_count"]
              .median().reset_index(name="cit_median_train"))
    out = []
    for part in (train, test):
        m = part.merge(med, left_on=["hour", "day_of_week"],
                       right_on=["hour_of_day", "weekday"], how="left")
        m["citations_hourly_median"] = m["cit_median_train"].fillna(0.0)
        m = m.drop(columns=["hour_of_day", "weekday", "cit_median_train"],
                   errors="ignore")
        out.append(m)
    return out[0], out[1]

## `build_block_aggregates()`

In [5]:
def build_block_aggregates(train):
    """
    Compute per-block historical means from training rows only.
    Holidays are excluded from the baseline means — observed-holiday Mondays
    (MLK, Presidents Day, etc.) were pulling weekday averages down and
    inflating error on regular Mondays by ~1.7 MAE.
    """
    clean = train[train["is_holiday"] == 0]
    global_mean = float(clean[TARGET].mean())

    block_mean = (clean.groupby(["lat", "lon"])[TARGET]
                       .mean().reset_index(name="block_mean"))
    block_hour_mean = (clean.groupby(["lat", "lon", "hour"])[TARGET]
                            .mean().reset_index(name="block_hour_mean"))
    block_hour_dow_mean = (clean.groupby(["lat", "lon", "hour", "day_of_week"])[TARGET]
                                .mean().reset_index(name="block_hour_dow_mean"))
    return {
        "global_mean": global_mean,
        "block_mean": block_mean,
        "block_hour_mean": block_hour_mean,
        "block_hour_dow_mean": block_hour_dow_mean,
    }

## `add_lag_features()`

In [6]:
def add_lag_features(df):
    """
    Per-block lag features. Generates all needed single-day shifts (1..7 for
    the rolling means, plus 14 and 28), materializes lag_3d_mean and
    lag_7d_mean, then drops the intermediate day columns not in the kept
    point-lag set. All shifts point strictly into the past — no leakage.
    """
    keys = ["lat", "lon", "timestamp"]
    kept_days = {h // 24 for h in LAG_HOURS}
    rolling_max = max(LAG_ROLLING_FEATURES.values())
    all_days = sorted(kept_days | set(range(1, rolling_max + 1)))

    for d in all_days:
        lag_name = f"lag_{d}d"
        shifted = df[["lat", "lon", "timestamp", TARGET]].copy()
        shifted["timestamp"] = shifted["timestamp"] + pd.Timedelta(days=d)
        shifted = shifted.rename(columns={TARGET: lag_name})
        df = df.merge(shifted, on=keys, how="left")

    for feat_name, n_days in LAG_ROLLING_FEATURES.items():
        cols = [f"lag_{d}d" for d in range(1, n_days + 1)]
        df[feat_name] = df[cols].mean(axis=1)

    drop_cols = [f"lag_{d}d" for d in all_days if d not in kept_days]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

## `apply_block_aggregates()`

In [7]:
def apply_block_aggregates(df, aggs):
    df = df.merge(aggs["block_mean"], on=["lat", "lon"], how="left")
    df = df.merge(aggs["block_hour_mean"], on=["lat", "lon", "hour"], how="left")
    df = df.merge(aggs["block_hour_dow_mean"],
                  on=["lat", "lon", "hour", "day_of_week"], how="left")
    gm = aggs["global_mean"]
    df["block_mean"] = df["block_mean"].fillna(gm)
    df["block_hour_mean"] = df["block_hour_mean"].fillna(df["block_mean"])
    df["block_hour_dow_mean"] = df["block_hour_dow_mean"].fillna(df["block_hour_mean"])
    return df

## `report()`

In [8]:
def report(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred) if len(y_true) > 1 else float("nan")
    print(f"  {label:<28} n={len(y_true):>8,}  MAE={mae:6.3f}  R²={r2:6.3f}")
    return {"n": int(len(y_true)), "mae": float(mae), "r2": float(r2)}

## Pipeline

In [9]:
print("=" * 60)
print("ParkCast SF — LightGBM Training (Accuracy-Optimized)")
print(f"MLflow: {MLFLOW_TRACKING_URI} · experiment={MLFLOW_EXPERIMENT}")
print("=" * 60)

with mlflow.start_run() as run:
    print(f"MLflow run: {run.info.run_id}")

    print("Loading processed data...")
    df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
    print(f"  Rows: {len(df):,}  Cols: {df.shape[1]}")

    print("Filtering to real meter-hour rows (target_is_estimated == 0)...")
    df = df[df["target_is_estimated"] == 0].reset_index(drop=True)
    print(f"  Real rows: {len(df):,}")

    print("Adding lag features (point + rolling means)...")
    df = add_lag_features(df)
    lag_names = [f"lag_{h//24}d" for h in LAG_HOURS] + list(LAG_ROLLING_FEATURES)
    lag_cov = {name: float(df[name].notna().mean()) for name in lag_names}
    print(f"  Lag coverage: {lag_cov}")

    print("Temporal 80/20 split...")
    train, test, split_time = temporal_split(df, frac=0.80)
    print(f"  Split at: {split_time}")
    print(f"  Train: {len(train):,}   Test: {len(test):,}")

    print("Refitting citations_hourly_median on train window only...")
    train, test = refit_citations_median(train, test, split_time)

    # Inner train/val split BEFORE building block aggregates so the aggregates
    # don't leak inner-val information into the features.
    train = train.sort_values("timestamp").reset_index(drop=True)
    val_cut = int(len(train) * 0.9)
    tr = train.iloc[:val_cut].copy()
    val_df = train.iloc[val_cut:].copy()
    print(f"  Inner split: train={len(tr):,}  val={len(val_df):,}")

    print("Computing block-level historical aggregates from inner-train only...")
    aggs = build_block_aggregates(tr)
    tr = apply_block_aggregates(tr, aggs)
    val_df = apply_block_aggregates(val_df, aggs)
    test = apply_block_aggregates(test, aggs)

    for part in (tr, val_df, test):
        part["neighborhood"] = part["neighborhood"].astype("category")

    # Residual target: y - baseline. Model learns only the deviation.
    X_tr = tr[FEATURES]
    y_tr = tr[TARGET].values - tr[BASELINE_COL].values
    X_val = val_df[FEATURES]
    y_val = val_df[TARGET].values - val_df[BASELINE_COL].values
    X_test = test[FEATURES]
    y_test = test[TARGET].values

    EVENT_WEIGHT = 20.0
    w_tr = np.where(tr["event_intensity"].values > 0, EVENT_WEIGHT, 1.0)
    w_val = np.where(val_df["event_intensity"].values > 0, EVENT_WEIGHT, 1.0)
    print(f"  Event rows (train): {(w_tr > 1).sum():,} weighted {EVENT_WEIGHT}×")

    # Recency decay — anchor at the end of the full outer-train window so
    # every train/val row has age ≥ 0 and weight ≤ 1. (Earlier bug used the
    # inner-train max, which made val rows get weight > 1.)
    if RECENCY_HALF_LIFE_DAYS is not None:
        anchor = train["timestamp"].max()
        tr_age_d = (anchor - tr["timestamp"]).dt.total_seconds().values / 86400.0
        val_age_d = (anchor - val_df["timestamp"]).dt.total_seconds().values / 86400.0
        recency_tr = 0.5 ** (tr_age_d / RECENCY_HALF_LIFE_DAYS)
        recency_val = 0.5 ** (val_age_d / RECENCY_HALF_LIFE_DAYS)
        w_tr = w_tr * recency_tr
        w_val = w_val * recency_val
        print(
            f"  Recency decay: half-life={RECENCY_HALF_LIFE_DAYS:.0f}d · "
            f"train age p5={np.percentile(tr_age_d, 5):.0f}d "
            f"p50={np.percentile(tr_age_d, 50):.0f}d "
            f"p95={np.percentile(tr_age_d, 95):.0f}d · "
            f"tr weight range [{recency_tr.min():.3f}, {recency_tr.max():.3f}]"
        )

    lgb_params = dict(
        n_estimators=4000,
        learning_rate=0.008,
        num_leaves=63,
        min_child_samples=100,
        feature_fraction=0.85,
        bagging_fraction=0.85,
        bagging_freq=5,
        reg_alpha=0.1,
        reg_lambda=0.1,
        objective="regression_l1",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )

    # Log run configuration to MLflow.
    mlflow.log_params({
        "target": TARGET,
        "baseline_col": BASELINE_COL,
        "n_features": len(FEATURES),
        "event_weight": EVENT_WEIGHT,
        "recency_half_life_days": RECENCY_HALF_LIFE_DAYS,
        "train_rows": int(len(tr)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test)),
        "split_time": str(split_time),
        "lag_hours": LAG_HOURS,
        "lag_rolling": LAG_ROLLING_FEATURES,
        **{f"lgb__{k}": v for k, v in lgb_params.items()},
    })
    mlflow.set_tags({
        "stage": "train",
        "model_family": "lightgbm-residual",
        "tracking_backend": "remote" if MLFLOW_TRACKING_URI.startswith("http") else "local",
    })
    mlflow.log_dict({"features": FEATURES, "numeric": FEATURES_NUMERIC,
                    "categorical": FEATURES_CATEGORICAL}, "features.json")
    mlflow.log_dict(lag_cov, "lag_coverage.json")

    print("Training LightGBM residual model...")
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y_tr,
        sample_weight=w_tr,
        eval_set=[(X_val, y_val)],
        eval_sample_weight=[w_val],
        eval_metric="mae",
        categorical_feature=["neighborhood"],
        callbacks=[lgb.early_stopping(150), lgb.log_evaluation(200)],
    )
    mlflow.log_metric("best_iteration", int(model.best_iteration_ or model.n_estimators))

    residual_pred = model.predict(X_test)
    preds = np.clip(test[BASELINE_COL].values + residual_pred, 0, 100)

    print("\nEvaluation on held-out temporal TEST set (real meter-hour rows):")
    metrics = {}
    baseline_preds = np.clip(test[BASELINE_COL].values, 0, 100)
    metrics["baseline"] = report(y_test, baseline_preds, "baseline: block×hr×dow mean")
    metrics["residual_model"] = report(y_test, preds, "baseline + residual model")

    ev_mask = test["event_intensity"].values > 0
    if ev_mask.any():
        print("\n  Event-row slice (event_intensity > 0):")
        metrics["baseline_events"] = report(
            y_test[ev_mask], baseline_preds[ev_mask], "baseline (events only)"
        )
        metrics["residual_events"] = report(
            y_test[ev_mask], preds[ev_mask], "residual model (events only)"
        )

    if metrics["baseline"]["mae"] <= metrics["residual_model"]["mae"]:
        print("\n  → Baseline wins.")
        metrics["winner"] = "baseline"
    else:
        print("\n  → Residual model wins.")
        metrics["winner"] = "residual_model"

    # Flatten metrics for MLflow (it wants scalars at known keys).
    for scope, vals in metrics.items():
        if not isinstance(vals, dict):
            continue
        for k in ("mae", "r2"):
            if k in vals:
                mlflow.log_metric(f"{scope}_{k}", float(vals[k]))

    print("\nTop feature importances:")
    imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
    for name, v in imp.head(15).items():
        print(f"  {name:<28} {int(v):>6}")
    mlflow.log_dict({name: int(v) for name, v in imp.items()}, "feature_importance.json")

    os.makedirs(MODEL_DIR, exist_ok=True)
    joblib.dump(model, MODEL_PATH)

    combined = (aggs["block_hour_dow_mean"]
                .merge(aggs["block_hour_mean"], on=["lat", "lon", "hour"], how="left")
                .merge(aggs["block_mean"], on=["lat", "lon"], how="left"))
    combined.attrs["global_mean"] = aggs["global_mean"]
    combined.to_parquet(BLOCK_AGG_PATH, index=False)

    meta = {
        "features": FEATURES,
        "target": TARGET,
        "split_time": str(split_time),
        "train_rows": int(len(tr)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test)),
        "best_iteration": int(model.best_iteration_ or model.n_estimators),
        "global_mean": aggs["global_mean"],
        "metrics": metrics,
        "mlflow_run_id": run.info.run_id,
        "mlflow_tracking_uri": MLFLOW_TRACKING_URI,
        "recency_half_life_days": RECENCY_HALF_LIFE_DAYS,
        "note": "Trained only on real meter-hour rows. Off-hour predictions "
                "are out of scope for this model.",
    }
    with open(META_PATH, "w") as f:
        json.dump(meta, f, indent=2)

    # Log + register model. Smart-promote: this version becomes @champion only
    # if its test MAE beats the incumbent @champion's (if any). Every run
    # creates a registry version; bad runs never silently replace good ones.
    mv = mlflow.lightgbm.log_model(
        model.booster_, name="model",
        registered_model_name=REGISTERED_MODEL_NAME,
    )
    this_mae = metrics["residual_model"]["mae"]
    client = mlflow.tracking.MlflowClient()
    try:
        champ = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "champion")
        champ_run = client.get_run(champ.run_id)
        champ_mae = float(champ_run.data.metrics.get("residual_model_mae", float("inf")))
    except Exception:
        champ_mae = float("inf")  # no champion yet → this run becomes one
    if this_mae < champ_mae:
        client.set_registered_model_alias(REGISTERED_MODEL_NAME, "champion", mv.registered_model_version)
        print(f"Promoted v{mv.registered_model_version} → @champion (MAE {this_mae:.4f} < {champ_mae:.4f})")
    else:
        print(f"Kept existing @champion (this MAE {this_mae:.4f} ≥ incumbent {champ_mae:.4f})")

    mlflow.log_artifact(BLOCK_AGG_PATH, artifact_path="block_aggs")
    mlflow.log_artifact(META_PATH, artifact_path="meta")

    print(f"\nSaved model: {MODEL_PATH}")
    print(f"Saved aggs:  {BLOCK_AGG_PATH}")
    print(f"Saved meta:  {META_PATH}")
    if MLFLOW_TRACKING_URI.startswith("http"):
        print(f"MLflow run:  {MLFLOW_TRACKING_URI}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}")
    else:
        print(f"MLflow run:  {MLFLOW_TRACKING_URI} · experiment={run.info.experiment_id} · run_id={run.info.run_id}")
        print("             (server was unreachable — backfill to remote with "
              "`mlflow experiments csv` + re-log when it's back)")
    print("=" * 60)

ParkCast SF — LightGBM Training (Accuracy-Optimized)
MLflow: http://34.134.159.106:5000 · experiment=parkcast-sf-lightgbm-v2
MLflow run: 3b96d9de4d5242d5b957fcbd1157b498
Loading processed data...


  Rows: 12,815,910  Cols: 33
Filtering to real meter-hour rows (target_is_estimated == 0)...


  Real rows: 5,637,410
Adding lag features (point + rolling means)...


  Lag coverage: {'lag_1d': 0.8775125456548308, 'lag_2d': 0.8723662107244284, 'lag_7d': 0.9748405739515132, 'lag_14d': 0.9562000989816245, 'lag_28d': 0.9210921327347132, 'lag_3d_mean': 0.9970266487624636, 'lag_7d_mean': 0.9970266487624636}
Temporal 80/20 split...


  Split at: 2026-01-27 18:00:00
  Train: 4,509,746   Test: 1,127,664
Refitting citations_hourly_median on train window only...


...


Top feature importances:
  block_hour_mean               11326
  temperature                   11304
  day_of_week                   11195
  neighborhood                  11140
  lag_7d_mean                    9179
  hour                           8523
  month                          7587
  lag_1d                         6696
  block_mean                     4674
  lag_28d                        3978
  sfpark_block_occ               3926
  lag_2d                         3900
  lat                            3773
  lon                            3375
  total_spaces                   3068


Successfully registered model 'parkcast-sf-lightgbm'.
2026/04/23 16:48:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: parkcast-sf-lightgbm, version 1


Created version '1' of model 'parkcast-sf-lightgbm'.


Promoted v1 → @champion (MAE 9.0533 < inf)



Saved model: /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/app/models/LightGBM.pkl
Saved aggs:  /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/app/models/LightGBM.block_aggs.parquet
Saved meta:  /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/app/models/LightGBM.meta.json
MLflow run:  http://34.134.159.106:5000/#/experiments/13/runs/3b96d9de4d5242d5b957fcbd1157b498
🏃 View run enchanting-sheep-743 at: http://34.134.159.106:5000/#/experiments/13/runs/3b96d9de4d5242d5b957fcbd1157b498
🧪 View experiment at: http://34.134.159.106:5000/#/experiments/13
